In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [29]:
curve_df = pd.read_hdf('./data/data.h5', key='curve_data')
sample_info = pd.read_hdf('./data/data.h5', key='sample_info')
sample_info = (sample_info
               .loc[~sample_info.sample_id.isin(['S1268904','S1268905'])])
igi_gene_call = pd.read_hdf('./data/data.h5', key='igi_gene_call')

join_df = (curve_df
            .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
            .merge(igi_gene_call, how='inner', on=['pcr_plate','sample_id','target']))
            

In [3]:
join_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
0,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,1,147103.828125,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
1,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,2,146864.656250,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
2,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,3,146410.109375,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
3,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,4,146188.328125,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
4,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,5,146078.421875,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined


In [4]:
join_df.columns

Index(['well_position', 'target', 'dye', 'amp_score', 'cq', 'threshold',
       'baseline_start', 'baseline_end', 'cycle_no', 'rn', 'drn', 'Fn',
       'pcr_plate', 'curve_idx', 'sample_id', 'sample_barcode', 'sample_type',
       'final_patient_result', 'current_sample_result', 'created_date',
       'record_type', 'retest_sample_id_1', 'retest_sample_id_2', 'file',
       'igi_call', 'thres_ct'],
      dtype='object')

In [30]:
clinical_ctrl_genes = (join_df
                       .loc[(join_df.sample_type == 'Clinical Sample') & (join_df.target.isin(['MS2','RnaseP'])), 
                            ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                       .copy())
clinical_ctrl_genes.loc[:,'groundtruth'] = 1

In [31]:
pos_ctrl_sample = (join_df
                   .loc[(join_df.sample_type == 'Positive Control (qPCR)'),
                        ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                   .copy())
pos_ctrl_sample.loc[:,'groundtruth'] = 1
pos_ctrl_sample.loc[pos_ctrl_sample.target.isin(['MS2','RnaseP']),'groundtruth'] = -1

In [32]:
(pos_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene   1             336
MS2     -1             133
N gene   1             469
ORF1ab   1             133
RnaseP  -1             336
S gene   1             133
Name: curve_idx, dtype: int64

In [33]:
human_ctrl_sample = (join_df
                     .loc[(join_df.sample_type == 'Human Normal Negative Control (Extraction)'),
                          ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                     .copy())
human_ctrl_sample.loc[:,'groundtruth'] = -1
human_ctrl_sample.loc[human_ctrl_sample.target.isin(['MS2','RnaseP']),'groundtruth'] = 1

In [34]:
(human_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             178
MS2      1              64
N gene  -1             242
ORF1ab  -1              64
RnaseP   1             178
S gene  -1              64
Name: curve_idx, dtype: int64

In [35]:
neg_ctrl_sample = (join_df
                   .loc[(join_df.sample_type == 'Negative Control (qPCR)'),
                        ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
                   .copy())
neg_ctrl_sample.loc[:,'groundtruth'] = -1

In [36]:
(neg_ctrl_sample
 .groupby(['target','groundtruth'])
 .curve_idx.nunique())

target  groundtruth
E gene  -1             336
MS2     -1             133
N gene  -1             469
ORF1ab  -1             133
RnaseP  -1             336
S gene  -1             133
Name: curve_idx, dtype: int64

In [37]:
# buffer_ctrl_sample = (join_df
#                       .loc[(join_df.sample_type == 'Buffer Negative Control (Extraction)') & (~join_df.target.isin(['MS2','RnaseP'])),
#                            ['curve_idx','target','sample_type','sample_id','igi_call','cycle_no','Fn','rn']]
#                       .copy())
# buffer_ctrl_sample.loc[:,'groundtruth'] = -1

In [38]:
# (buffer_ctrl_sample
#  .groupby(['target','groundtruth'])
#  .curve_idx.nunique())

In [39]:
groundtruth_df = pd.concat([neg_ctrl_sample, pos_ctrl_sample, 
                            human_ctrl_sample, clinical_ctrl_genes])

In [40]:
final_patient_df = (groundtruth_df[['sample_id','sample_type']]
 .drop_duplicates()
 .merge(join_df[['sample_id','final_patient_result']].drop_duplicates()))
 

In [41]:
join_df.loc[(join_df.sample_type == 'Clinical Sample') & (join_df.final_patient_result.isna()),'pcr_plate'].drop

<bound method Series.drop of 437760     AC00DAS2
437761     AC00DAS2
437762     AC00DAS2
437763     AC00DAS2
437764     AC00DAS2
             ...   
3147105    C302NBFG
3147106    C302NBFG
3147107    C302NBFG
3147108    C302NBFG
3147109    C302NBFG
Name: pcr_plate, Length: 50240, dtype: object>

In [42]:
join_df.loc[(join_df.sample_type == 'Clinical Sample') & (join_df.final_patient_result.isna()),['sample_id','sample_barcode', 'pcr_plate']].drop_duplicates()

,sample_id,sample_barcode,pcr_plate
437760,S146249,NPSWAB0093-Retest 1,AC00DAS2
440160,S146250,NPSWAB0191-Retest 2,AC00DAS2
996740,S139620,UHSS0000006155- Retest 1,AC00DBSX
997060,S139632,LLIGI0000031244- Retest 1,AC00DBSX
1004100,S139621,UHSS0000006153- Retest 1,AC00DBSX
...,...,...,...
3145350,S117365,LLIGI0000035491- Retest 1,C302NBFG
3145670,S117358,LLIGI0000035465- Retest 1,C302NBFG
3146310,S117391,LLIGI0000032851- Retest 1,C302NBFG
3146630,S117372,LLIGI0000032847- Retest 1,C302NBFG


In [43]:
final_patient_df.loc[(final_patient_df.sample_type == 'Negative Control (qPCR)') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Negative'
final_patient_df.loc[(final_patient_df.sample_type == 'Positive Control (qPCR)') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Positive'
final_patient_df.loc[(final_patient_df.sample_type == 'Human Normal Negative Control (Extraction)') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Negative'
final_patient_df.loc[(final_patient_df.sample_type == 'Clinical Sample') & (final_patient_df.final_patient_result.isna()), 'final_patient_result'] = 'Retest'

In [44]:
data_ids = final_patient_df.sample_id
labels = [1]*len(data_ids)
stratified_col = final_patient_df.final_patient_result


In [45]:

# stratified_col = [groundtruth_df.target[i] + '_' + groundtruth_df.groundtruth[i] for i in range(len()) ]
train_ids, test_ids, train_labels, test_labels = train_test_split(data_ids, stratified_col, test_size=0.25, stratify=stratified_col, random_state=1)
train_ids, val_ids, train_labels, val_labels = train_test_split(train_ids, train_labels, test_size=0.25, stratify=final_patient_df[final_patient_df.sample_id.isin(train_ids)].final_patient_result, random_state=1)

In [46]:
groundtruth_df.loc[:,'split'] = 'train'
groundtruth_df.loc[groundtruth_df.sample_id.isin(test_ids), 'split'] = 'test'
groundtruth_df.loc[groundtruth_df.sample_id.isin(val_ids), 'split'] = 'val'


In [48]:
(groundtruth_df
 .groupby(['split','groundtruth'])
 .curve_idx.nunique())

split  groundtruth
test   -1               623
        1              5552
train  -1              1481
        1             12468
val    -1               453
        1              4179
Name: curve_idx, dtype: int64

In [49]:
groundtruth_df.to_csv('./data/groundtruth_df.csv', index = False)